# Final Project Phase 2 Summary
This Jupyter Notebook (.ipynb) will serve as the skeleton file for your submission for Phase 2 of the Final Project. Answer all statements addressed below as specified in the instructions for the project, covering all necessary details. Please be clear and concise in your answers. Each response should be at most 3 sentences. Good luck! <br><br>

Note: To edit a Markdown cell, double-click on its text.

## Jupyter Notebook Quick Tips
Here are some quick formatting tips to get you started with Jupyter Notebooks. This is by no means exhaustive, and there are plenty of articles to highlight other things that can be done. We recommend using HTML syntax for Markdown but there is also Markdown syntax that is more streamlined and might be preferable.
<a href = "https://towardsdatascience.com/markdown-cells-jupyter-notebook-d3bea8416671">Here's an article</a> that goes into more detail. (Double-click on cell to see syntax)

# Heading 1
## Heading 2
### Heading 3
#### Heading 4
<br>
<b>BoldText</b> or <i>ItalicText</i>
<br> <br>
Math Formulas: $x^2 + y^2 = 1$
<br> <br>
Line Breaks are done using br enclosed in < >.
<br><br>
Hyperlinks are done with: <a> https://www.google.com </a> or
<a href="http://www.google.com">Google</a><br>

# Data Collection and Cleaning
You are required to provide data collection and cleaning for the three (3) minimum datasets. Create a function for each of the following sections that reads or scrapes data from a file or website, manipulate and cleans the parsed data, and writes the cleaned data into a new file.

Make sure your data cleaning and manipulation process is not too simple. Performing complex manipulation and using modules not taught in class shows effort, which will increase the chance of receiving full credit.


## Data Sources
Include sources (as links) to your datasets. Add any additional data sources if needed. Clearly indicate if a data source is different from one submitted in your Phase I, as we will check that it satisfies the requirements.
*   Downloaded Dataset Source: https://data-nces.opendata.arcgis.com/datasets/nces::public-school-characteristics-2022-23/explore
*   Web Collection #1 Source: https://en.wikipedia.org/wiki/List_of_United_States_counties_by_per_capita_income
*   Web Collection #2 Source: https://api.census.gov/data/2021/pep/populationget=POP_2021,NAME&for=state:*&key=961d7e26357d056e8ce66fdfc33052149c4dff2c"
 



## Downloaded Dataset Requirement

Fill in the predefined functions with your data scraping/parsing code. You may modify/rename each function as you seem fit, but you must provide at least 3 separate functions that clean each of your required datasets.


In [17]:
import pandas as pd
import numpy as np

def data_parser():
  original_file = pd.read_csv("Public_School_Characteristics_2022-23.csv")

  #filter dataframe to only columns we will focus on
  correct_header_file = original_file[["STABR", "SCH_NAME", "LSTATE", "SCHOOL_LEVEL", "NMCNTY", "TOTFRL", "TOTMENROL", "TOTFENROL", "TOTAL", "MEMBER", "FTE", "STUTERATIO"]]
  #Inconsistency: STABR and LSTATE are the same (abbreviations of states)
  #Inconsistency: TOTAL and MEMBER both give same total amount of students as a float
  
  correct_header_file = correct_header_file.drop(["LSTATE", "MEMBER"], axis = 1) #axis = 1 --> columns
  #Change column titles (headers) to make more sense
  correct_header_file = correct_header_file.rename(columns = {"STABR" : "State", "LEA_NAME": "School District", "SCH_NAME": "School Name", "SCHOOL_LEVEL": "School Level", "NMCNTY": "County Name", "TOTFRL": "Free/Reduced Lunch", "TOTMENROL": "Male Students", "TOTFENROL": "Female Students", "TOTAL": "Total Students", "FTE": "Full-Time Teachers", "STUTERATIO": "Student-to-Teacher Ratio"})

  #We want to focus on only GA schools
  GA_info = correct_header_file[correct_header_file["State"] == "GA"].reset_index(drop = True) #reset indexes to 0,1,2, ... - drop older indeces

  #Inconsistency: in the "Full-Time Teachers" column many values have decimal values, there can not be .5 of a teacher
  #round teachers down to the nearest integer
  #Inconsistency: Many NaN in "Full-Time Teachers"
        #anywhere that teachers is NaN, replace with average teachers from that county

  county_teacher_avg = GA_info.groupby("County Name")["Full-Time Teachers"].mean()
  GA_info["Full-Time Teachers"] = np.where(GA_info["Full-Time Teachers"].isna(), county_teacher_avg[GA_info["County Name"]], GA_info["Full-Time Teachers"])

  GA_info["Full-Time Teachers"] = np.floor(GA_info["Full-Time Teachers"]).astype(int) #rounds teachers count down to closest integer
  #update Male Students, Female Students, and Total Students to integers as well for consistency
  #Inconsistency: NaN in these columns
        #Replace with average of that group like we did for Teachers

  county_mStudent_avg = GA_info.groupby("County Name")["Male Students"].mean()
  GA_info["Male Students"] = np.where(GA_info["Male Students"].isna(), county_mStudent_avg[GA_info["County Name"]], GA_info["Male Students"])
  GA_info["Male Students"] = np.floor(GA_info["Male Students"]).astype(int)

  county_fStudent_avg = GA_info.groupby("County Name")["Female Students"].mean()
  GA_info["Female Students"] = np.where(GA_info["Female Students"].isna(), county_mStudent_avg[GA_info["County Name"]], GA_info["Female Students"])
  GA_info["Female Students"] = np.floor(GA_info["Female Students"]).astype(int)

  county_tStudent_avg = GA_info.groupby("County Name")["Total Students"].mean()
  GA_info["Total Students"] = np.where(GA_info["Total Students"].isna(), county_mStudent_avg[GA_info["County Name"]], GA_info["Total Students"])
  GA_info["Total Students"] = np.floor(GA_info["Total Students"]).astype(int)

  #Now lets re-calculate the Student-to-Teacher Ratio - Round to 2 decimal places
  GA_info["Student-to-Teacher Ratio"] = round((GA_info["Total Students"] / GA_info["Full-Time Teachers"]), 2)

  #For our data collection, we will focus on Georgia Counties and their rankings
  #Meaning we want to group everything by district and use County averages for calculations

  #First - Some rows have negative values in the Free/Reduced Lunch Category
      #Replace with county average

  #If negative value then replace with NaN
  GA_info["Free/Reduced Lunch"] = np.where((GA_info["Free/Reduced Lunch"] < 0), np.nan, GA_info["Free/Reduced Lunch"])

  county_lunch_avg = GA_info.groupby("County Name")["Free/Reduced Lunch"].mean()
  GA_info["Free/Reduced Lunch"] = np.where(GA_info["Free/Reduced Lunch"].isna(), round(county_lunch_avg[GA_info["County Name"]], 0).astype(int), GA_info["Free/Reduced Lunch"])

  #We can drop School Name and School Level as these values cannot be averaged
  GA_info = GA_info.drop(["School Name", "School Level"], axis = 1)

  #Now group by counties:
  GA_counties = GA_info.groupby("County Name")

  #calculate the mean of Free/Reduced Lunches per county
  #Number of Male, Female, and Total students in each county
  #Avg teachers per school and avg student to teacher ratio per school in each county
  GA_counties = GA_counties.agg({"Free/Reduced Lunch": "mean", "Male Students": "sum", "Female Students": "sum", "Total Students": "sum", "Full-Time Teachers": "mean", "Student-to-Teacher Ratio": "mean"}).round(2)

  GA_counties.to_csv("GA_County_Info.csv", index = True)
  return GA_counties.head(50)





############ Function Call ############
data_parser()

/var/folders/zy/w5ds3w812l31dl9lvdd9zl7h0000gn/T/ipykernel_45645/3057716498.py:5: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  original_file = pd.read_csv("Public_School_Characteristics_2022-23.csv")


,Free/Reduced Lunch,Male Students,Female Students,Total Students,Full-Time Teachers,Student-to-Teacher Ratio
County Name,,,,,,
Appling County,554.60,1747,1716,3463,50.80,13.59
Atkinson County,347.75,825,805,1630,29.25,13.72
Bacon County,412.25,1096,1032,2128,40.25,13.15
Baker County,292.00,162,138,300,13.00,9.84
Baldwin County,599.71,2346,2356,4702,52.86,12.85
Banks County,360.20,1685,1564,3249,49.40,13.10
Barrow County,478.56,7594,7288,14882,65.00,14.12
Bartow County,421.39,9322,8981,18303,51.78,14.99
Ben Hill County,491.00,1528,1475,3003,44.60,13.93


## Web Collection Requirement \#1


In [13]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def web_parser1(url):
  agent = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/97.0.4692.99 Safari/537.36"} # i use the same agent used in the homework lol
  response = requests.get(url, headers = agent)
  soup = BeautifulSoup(response.text, 'html.parser') #get our html so that we can easily traverse it

  table_tag = soup.find('table', {"class" : "wikitable"})

  rows = []
  for r in table_tag.find_all('tr')[1:]: #skip the header row
    td_tags = r.find_all('td') #effectivly skips the rank header column, we dont need that

    if len(td_tags) == 7: #some of the columns were for the state overall and contained less columns, so dont include those
      state = td_tags[1].text.strip()

      if state == "Georgia": #make sure the state is Georgia
        county = td_tags[0].text.strip()
        per_cap = float(td_tags[2].text.strip()[1:].replace(',',''))
        median_house = float(td_tags[3].text.strip()[1:].replace(',','')) #remove dollar signs and replaced commas with decimal points, then concerted to floats for easy analysis
        rows.append([county, per_cap, median_house])

  df = pd.DataFrame(rows, columns = ['County', 'Per Capita Income (USD)', 'Median Household Income (USD)'])

  df.to_csv("GA_County_Income.csv", index = False)
  return df.head(10)


############ Function Call ############
web_parser1('https://en.wikipedia.org/wiki/List_of_United_States_counties_by_per_capita_income')

,County,Per Capita Income (USD),Median Household Income (USD)
0,Fulton,36757.0,56857.0
1,Fayette,35344.0,79977.0
2,Forsyth,34582.0,86569.0
3,Oconee,33801.0,75004.0
4,Cobb,33069.0,63920.0
5,Harris,31482.0,69223.0
6,Columbia,30949.0,69306.0
7,Cherokee,29615.0,67261.0
8,DeKalb,28810.0,50856.0
9,Greene,27828.0,42565.0


## Web Collection Requirement \#2

In [18]:
#for this dataset we will take information from an API site on Public School Data by county and calculate state and then nationwide averages
#this is so we can compare the stats for GA public schools to US public school stats
import requests, re, json
import pandas as pd
import numpy as np

def web_parser2():
  url = "https://api.census.gov/data/2021/pep/population?get=POP_2021,NAME&for=state:*&key=961d7e26357d056e8ce66fdfc33052149c4dff2c"
  response = requests.get(url).json() #get population by state info from API

  data = pd.DataFrame(response) #put information into pandas dataframe
  data.columns = data.iloc[0] #set first row to be the header
  data = data[1:].reset_index(drop = True)

  data_clean = data.drop(["state"], axis = 1) #drop state column as it contains states represented by their number -- axis = 1 represents columns
  data_clean = data_clean.rename(columns = {"POP_2021": "Population", "NAME": "State"}) #rename columns
  data_clean = data_clean.sort_values("State").reset_index(drop=True) #Sort alphebetically by state name to make it easier to read - reset index

  data_clean.to_csv("State_Pop.csv", index = True)
  return data_clean





############ Function Call ############
web_parser2()

,Population,State
0,5039877,Alabama
1,732673,Alaska
2,7276316,Arizona
3,3025891,Arkansas
4,39237836,California
5,5812069,Colorado
6,3605597,Connecticut
7,1003384,Delaware
8,670050,District of Columbia
9,21781128,Florida


## Additional Dataset Parsing/Cleaning Functions

Write any supplemental (optional) functions here.

In [15]:
import pandas as pd
import numpy as np

def extra_source1():
    original_file = pd.read_csv("Public_School_Characteristics_2022-23.csv")
    #filter dataframe to only columns we will focus on
    correct_header_file = original_file[["STABR", "TOTFRL", "TOTAL", "FTE", "STUTERATIO"]] #we will group by state to find state averages
    #rename colums for consistency
    correct_header_file = correct_header_file.rename(columns = {"STABR" : "State", "TOTFRL": "Free/Reduced Lunch", "TOTAL": "Total Students", "FTE": "Full-Time Teachers", "STUTERATIO": "Student-to-Teacher Ratio"})

    #replace any missing data with NaN then drop NaN rows
    clean_file = correct_header_file.replace([-1, 0], np.nan) #replace with Nan
    clean_file.dropna(axis = 0, inplace =True) #drops rows with Nan's inplace

    #group by state and take average/sum of each column
    US_states = clean_file.groupby("State")
    US_states = US_states.agg({"Free/Reduced Lunch": "mean", "Total Students": "sum", "Full-Time Teachers": "sum", "Student-to-Teacher Ratio": "mean"}).round(2)

    US_states.to_csv("US_State_Info.csv", index = True)
    return US_states


############ Function Call ############
extra_source1()

/var/folders/zy/w5ds3w812l31dl9lvdd9zl7h0000gn/T/ipykernel_45645/1524738470.py:5: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  original_file = pd.read_csv("Public_School_Characteristics_2022-23.csv")


,Free/Reduced Lunch,Total Students,Full-Time Teachers,Student-to-Teacher Ratio
State,,,,
AK,104.25,130709.0,7160.48,20.22
AL,327.11,733241.0,41775.44,17.63
AR,296.27,492617.0,38748.81,13.38
AZ,319.52,880784.0,48974.62,17.35
CA,349.94,5837122.0,267284.85,21.22
CO,193.57,795906.0,48841.14,15.92
CT,208.00,495426.0,42142.68,11.99
FL,391.96,2817201.0,155837.73,18.20
GA,454.62,1750652.0,118650.80,14.36


In [16]:
# Define further extra source functions as necessary

# Inconsistencies

For each inconsistency (NaN, null, duplicate values, empty strings, etc.) you discover in your datasets, write at least 2 sentences stating the significance, how you identified it, and how you handled it.

1. Webcollection 1: HTML - Found unnnecessary characters from a string + the data type was not optimal for data analysis for the table information of per capita income and median family income. Both column values contained $ signs and were typed as strings. To fix this, the string values were sliced to not include the first character [1:] and then the whole string was casted to a float.

2. Downloaded Dataset: CSV File - Columns such as total teachers and total students were expressed as decimals, but there cannot be a half of student or teacher. To fix this inconsistancy, the values for these columns were rounded down using by casting to an int.

3. Downloaded Dataset: CSV File - Missing values/invalid values are in multiple columns of the dataset for several different public schools ( ex. Negative values for # of Free/Reduced Lunch). To combat this inconsistancy and get accurate data analysis, we altered the missing values with the average value for that column.

4. Downloaded Dataset + Webcollection 1: Changed CSV names for school county to be consistant with other data sets (CNTY to County/County name)

5. (if applicable)
